In [73]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time
import pandas as pd
import re

In [74]:
# Get links per town per city in Metro Manila

driver = webdriver.Chrome('C:/Users/Leibniz/Downloads/chromedriver-win64/chromedriver.exe')
url = "https://www.lamudi.com.ph/metro-manila/buy/"
driver.get(url)

# Initialize lists to store links, cities, and towns
links = []
cities = []
towns = []

# Extract the links of specific locations in Metro Manila
specific_element = driver.find_element(By.XPATH, '/html/body/div[4]/div[2]/div[1]/div[2]')
a_tags = specific_element.find_elements(By.TAG_NAME, 'a')

# Print out the number of links found
print(f"Found {len(a_tags)} links on page {url}")

for a_tag in a_tags:
    link = a_tag.get_attribute('href')
    if link:
        s = re.sub('-', " ", re.split("/", link)[4])
        def convert_into_uppercase(a):
            return a.group(1) + a.group(2).upper()
        city = re.sub("(^|\s)(\S)", convert_into_uppercase, s)

        town = a_tag.get_attribute('data-label')

        if link not in links and link != "https://www.lamudi.com.ph/" and link != "https://www.lamudi.com.ph/metro-manila/buy/":
            links.append(link)
            cities.append(city)
            towns.append(town)

# Close the driver
driver.quit()

C:\Users\Leibniz\AppData\Local\Temp\ipykernel_28772\270643410.py:3: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome('C:/Users/Leibniz/Downloads/chromedriver-win64/chromedriver.exe')


Found 120 links on page https://www.lamudi.com.ph/metro-manila/buy/


In [75]:
# Create DataFrame
pd.set_option('display.max_colwidth', None)
towns_df = pd.DataFrame({'Link': links, 'City': cities, 'Town': towns})

# Save DataFrame to a CSV file
town_links.to_csv('housing_urls.csv', index=False)

print("Links saved to housing_urls.csv")

Links saved to housing_urls.csv


In [76]:
towns_df

,Link,City,Town
0,https://www.lamudi.com.ph/metro-manila/quezon-city/fairview/buy/,Quezon City,Fairview
1,https://www.lamudi.com.ph/metro-manila/quezon-city/cubao-1/buy/,Quezon City,Cubao
2,https://www.lamudi.com.ph/metro-manila/quezon-city/tandang-sora-3/buy/,Quezon City,Tandang Sora
3,https://www.lamudi.com.ph/metro-manila/quezon-city/eastwood-city-1/buy/,Quezon City,Eastwood City
4,https://www.lamudi.com.ph/metro-manila/quezon-city/batasan-hills/buy/,Quezon City,Batasan Hills
...,...,...,...
114,https://www.lamudi.com.ph/metro-manila/pasig/rosario-14/buy/,Pasig,Rosario
115,https://www.lamudi.com.ph/metro-manila/pasig/kapitolyo/buy/,Pasig,Kapitolyo
116,https://www.lamudi.com.ph/metro-manila/pasig/bagong-ilog/buy/,Pasig,Bagong Ilog
117,https://www.lamudi.com.ph/metro-manila/pasig/santolan/buy/,Pasig,Santolan


In [146]:
# Get information of all properties per town

# Create a new instance of Chrome driver
driver = webdriver.Chrome('C:/Users/Leibniz/Downloads/chromedriver-win64/chromedriver.exe')

# URL of the webpage to scrape
base_url = "https://www.lamudi.com.ph/metro-manila/pasig/ortigas-cbd-1/buy/"
url = base_url

# Initialize lists to store property details
property_titles = []
property_prices = []
property_categories = []
property_bedroom_nums = []
property_bathroom_nums = []
property_floor_areas = []
property_geo_locations = []

# Get the number of pages
driver.get(url)
number_pages = driver.find_element(By.CLASS_NAME, 'BaseSection.Pagination').get_attribute('data-pagination-end')

# Loop through each page
for i in range(1, int(number_pages) + 1):
    url = base_url + "?page=" + str(i)

    # Open the webpage
    driver.get(url)

    # Wait for the overlay to disappear
    WebDriverWait(driver, 10).until(EC.invisibility_of_element_located((By.CLASS_NAME, 'blockUI blockOverlay')))

    # Extract property details
    property_elements = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.XPATH, '//div[@class="ListingCell-AllInfo ListingUnit"]'))
    )

    for element in property_elements:
        # Extract Information
        title = element.find_element(By.CLASS_NAME, 'ListingCell-KeyInfo-title').text
        price = element.get_attribute('data-price')
        category = element.get_attribute('data-category')
        bedroom_num = element.get_attribute('data-bedrooms')
        bathroom_num = element.get_attribute('data-bathrooms')
        floor_area = element.get_attribute('data-building_size')
        geo_location = element.get_attribute('data-geo-point')
        
        # Append to corresponding lists
        property_titles.append(title)
        property_prices.append(price)
        property_categories.append(category)
        property_bedroom_nums.append(bedroom_num)
        property_bathroom_nums.append(bathroom_num)
        property_floor_areas.append(floor_area)
        property_geo_locations.append(geo_location)

# Close the driver
driver.quit()

C:\Users\Leibniz\AppData\Local\Temp\ipykernel_28772\2154961800.py:9: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome('C:/Users/Leibniz/Downloads/chromedriver-win64/chromedriver.exe')


In [147]:
# Create a DataFrame to store the scraped data
df = pd.DataFrame({
    'Property Title': property_titles,
    'Price': property_prices,
    'Category': property_categories,
    'Num_Bedrooms': property_bedroom_nums,
    'Num_Bathrooms': property_bathroom_nums,
    'Floor_Area': property_floor_areas,
    'Geo_Locations': property_geo_locations
})

# Save DataFrame to a CSV file
df.to_csv('property_info.csv', index=False)

print("Scraping completed. Property information saved to property_info.csv.")

Scraping completed. Property information saved to property_info.csv.


In [171]:
# Get information of all properties per town

def properties_info(base_url):
    '''
    Based on base_url, gets all property info of the town and returns it on a data frame.
    '''
    # Create a new instance of Chrome driver
    driver = webdriver.Chrome('C:/Users/Leibniz/Downloads/chromedriver-win64/chromedriver.exe')

    # Initialize lists to store property details
    property_titles = []
    property_prices = []
    property_categories = []
    property_bedroom_nums = []
    property_bathroom_nums = []
    property_floor_areas = []
    property_geo_locations = []

    # Get the number of pages
    driver.get(base_url)
    number_pages = driver.find_element(By.CLASS_NAME, 'BaseSection.Pagination').get_attribute('data-pagination-end')

    # Loop through each page
    for i in range(1, int(number_pages) + 1):
        url = base_url + "?page=" + str(i)

        driver.get(url)

        # Wait for the overlay to disappear
        WebDriverWait(driver, 10).until(EC.invisibility_of_element_located((By.CLASS_NAME, 'blockUI blockOverlay')))

        # Extract property details
        property_elements = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.XPATH, '//div[@class="ListingCell-AllInfo ListingUnit"]'))
        )

        for element in property_elements:
            # Extract Information
            title = element.find_element(By.CLASS_NAME, 'ListingCell-KeyInfo-title').text
            price = element.get_attribute('data-price')
            category = element.get_attribute('data-category')
            bedroom_num = element.get_attribute('data-bedrooms')
            bathroom_num = element.get_attribute('data-bathrooms')
            floor_area = element.get_attribute('data-building_size')
            geo_location = element.get_attribute('data-geo-point')

            # Append to corresponding lists
            property_titles.append(title)
            property_prices.append(price)
            property_categories.append(category)
            property_bedroom_nums.append(bedroom_num)
            property_bathroom_nums.append(bathroom_num)
            property_floor_areas.append(floor_area)
            property_geo_locations.append(geo_location)

    driver.quit()
    
    # Create a DataFrame to store the scraped data
    df = pd.DataFrame({
        'Property Title': property_titles,
        'Price': property_prices,
        'Category': property_categories,
        'Num_Bedrooms': property_bedroom_nums,
        'Num_Bathrooms': property_bathroom_nums,
        'Floor_Area': property_floor_areas,
        'Geo_Locations': property_geo_locations
    })
    
    df = df.drop_duplicates()

    # Save DataFrame to a CSV file
    city = towns_df[towns_df['Link'] == base_url]['City'].iloc[0]
    town = towns_df[towns_df['Link'] == base_url]['Town'].iloc[0]
    file = f'property_info-{city}-{town}.csv'
    df.to_csv(file, index=False)

    print("Scraping completed. Property information saved to " + file)
    print(len(df))
    
properties_info("https://www.lamudi.com.ph/metro-manila/pasig/ortigas-cbd-1/buy/")

C:\Users\Leibniz\AppData\Local\Temp\ipykernel_28772\2681923660.py:8: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome('C:/Users/Leibniz/Downloads/chromedriver-win64/chromedriver.exe')


Scraping completed. Property information saved to property_info-Pasig-Ortigas CBD.csv
933


In [177]:
df_check = pd.read_csv('property_info-Pasig-Ortigas CBD.csv')
df_check

,Property Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Geo_Locations
0,"1BR Regular Condo Unit for Sale in Ortigas CBD, Pasig | Residences at The Galleon (42...",26795000.0,condominium,1.0,1.0,69.0,"[121.0598601955,14.5880946859]"
1,"1BR Regular Condo Unit for Sale in Ortigas CBD, Pasig | Residences at The Galleon (25...",24987000.0,condominium,1.0,1.0,74.0,"[121.0598601955,14.5880946859]"
2,"1BR Regular Condo Unit for Sale in Ortigas CBD, Pasig | Residences at The Galleon (30...",25355000.0,condominium,1.0,1.0,70.0,"[121.0598601955,14.5880946859]"
3,"1BR Regular Condo Unit for Sale in Ortigas CBD, Pasig | Residences at The Galleon (19...",26202000.0,condominium,1.0,1.0,69.0,"[121.0598601955,14.5880946859]"
4,"1BR Regular Condo Unit for Sale in Ortigas CBD, Pasig | Residences at The Galleon (36...",25560000.0,condominium,1.0,1.0,70.0,"[121.0598601955,14.5880946859]"
...,...,...,...,...,...,...,...
928,Studio for Sale in Ortigas Center Sapphire Bloc,11888.0,condominium,1.0,1.0,28.0,"[121.059675,14.583771]"
929,Sonata Private Residences 1 Bedroom with 1 Parking Slot,7300000.0,condominium,1.0,1.0,49.0,"[121.0585,14.58119]"
930,Modern and Elegant 2 Bedroom Condo Unit for Sale in St. Francis Shangri-La Place,35000000.0,condominium,2.0,2.0,119.0,"[121.059675,14.583771]"
931,1 Bedroom Spacious Condo Unit for Sale in St. Francis Shangri-La Place,16000000.0,condominium,1.0,1.0,62.0,"[121.059675,14.583771]"


In [179]:
df_check.groupby('Category').mean()

,Price,Num_Bedrooms,Num_Bathrooms,Floor_Area
Category,,,,
apartment,1.312500e+07,6.000000,6.000000,200.000000
commercial,6.108776e+07,NaN,0.000000,323.991991
condominium,2.409262e+07,1.574648,1.540029,92.330945
house,1.705000e+07,4.000000,3.666667,220.358333
land,8.643600e+08,NaN,NaN,NaN


In [ ]:
batch_size = 10  # Set the batch size as needed

# Get the total number of links
total_links = len(towns_df['Link'])

# Iterate over the links in batches
for i in range(0, total_links, batch_size):
    # Get the current batch of links
    batch_links = towns_df['Link'].iloc[i:i+batch_size]
    
    # Process each link in the batch
    for link in batch_links:
        try:
            properties_info(link)
        except Exception as e:
            print(f"Error processing link {link}: {e}")

In [148]:
len(df)

1032

In [149]:
df

,Property Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Geo_Locations
0,"1BR Executive Condo Unit for Sale in The Sapphire Bloc – East, Ortigas CBD, Pasig",12300000,condominium,1,1,49.5,"[121.0625309,14.5876677]"
1,"1BR Condo Unit for Sale in The Sapphire Bloc – East, Ortigas CBD, Pasig | 36sqm",9900000,condominium,1,1,36,"[121.0625309,14.5876677]"
2,LOFT in ORTIGAS East Of Galleria - Spacious One Bedroom for sale inside CBD,4500000,condominium,1,1,40.34,"[121.059675,14.583771]"
3,"2BR Condo Unit for Sale in Ortigas CBD, Pasig | Residences at The Galleon (29E)",39150000,condominium,2,2,114,"[121.0598601955,14.5880946859]"
4,"Office Condo for Sale in Ortigas CBD, Pasig | Offices at The Galleon (35F), 111sqm",35356000,commercial,None,None,111,"[121.0599336955,14.5881146029]"
...,...,...,...,...,...,...,...
1027,1 BR Condo Rush Sale in Ortigas Center Pasig,9119334,condominium,1,1,36,"[121.059675,14.583771]"
1028,1 One Bedroom Condominium Unit in Ortigas Center Pasig,12900000,condominium,1,1,49,"[121.059675,14.583771]"
1029,"For Sale: 4 Bedroom Semi-furnished at The Crescent in Ortigas Center, Pasig City",28200000,condominium,4,3,367.5,"[121.059675,14.583771]"
1030,"For Sale: Fully Renovated 3 Bedroom Unit at AIC Gold Tower in Ortigas CBD, Pasig",18000000,condominium,3,2,167,"[121.059675,14.583771]"


In [153]:
df.drop_duplicates()

,Property Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Geo_Locations
0,"1BR Executive Condo Unit for Sale in The Sapphire Bloc – East, Ortigas CBD, Pasig",12300000,condominium,1,1,49.5,"[121.0625309,14.5876677]"
1,"1BR Condo Unit for Sale in The Sapphire Bloc – East, Ortigas CBD, Pasig | 36sqm",9900000,condominium,1,1,36,"[121.0625309,14.5876677]"
2,LOFT in ORTIGAS East Of Galleria - Spacious One Bedroom for sale inside CBD,4500000,condominium,1,1,40.34,"[121.059675,14.583771]"
3,"2BR Condo Unit for Sale in Ortigas CBD, Pasig | Residences at The Galleon (29E)",39150000,condominium,2,2,114,"[121.0598601955,14.5880946859]"
4,"Office Condo for Sale in Ortigas CBD, Pasig | Offices at The Galleon (35F), 111sqm",35356000,commercial,None,None,111,"[121.0599336955,14.5881146029]"
...,...,...,...,...,...,...,...
1022,Studio for Sale in Ortigas Center Sapphire Bloc,11888,condominium,1,1,28,"[121.059675,14.583771]"
1023,Sonata Private Residences 1 Bedroom with 1 Parking Slot,7300000,condominium,1,1,49,"[121.0585,14.58119]"
1024,Modern and Elegant 2 Bedroom Condo Unit for Sale in St. Francis Shangri-La Place,35000000,condominium,2,2,119,"[121.059675,14.583771]"
1025,1 Bedroom Spacious Condo Unit for Sale in St. Francis Shangri-La Place,16000000,condominium,1,1,62,"[121.059675,14.583771]"
